# 05 · External restaurant records and extraction evaluation

The notebook uses a **synthetic Overpass-format snapshot** and the existing synthetic text-ingestion batch. It makes no network calls. The Python package performs all validation, import, and evaluation; notebook cells only invoke those functions and inspect results.

Learning objectives: distinguish OSM source identities from real-world establishments, inspect the duplicate-review queue, and interpret exact-match field metrics.

In [ ]:
import json
from collections import Counter
from copy import deepcopy
from pathlib import Path

from savoroute.ingestion.osm import import_osm_snapshot
from savoroute.data.catalog import load_catalog
from savoroute.services.discovery import DiscoveryService
from savoroute.retrieval.lexical import SearchFilters
from savoroute.ingestion.sources import parse_source_batch
from savoroute.ingestion.extraction import LabeledTextExtractor
from savoroute.services.ingestion import ingest_sources
from savoroute.evaluation.extraction import evaluate_extraction

SAMPLES = Path("data/sample")

## 1. Inspect an external-source snapshot

Overpass returns OSM elements (nodes, ways, or relations) with tags. `name` and `cuisine` are recorded tags; they are not independently verified facts. This fixture uses fabricated OSM IDs and a marker preventing accidental attribution as live OSM data.

In [ ]:
snapshot = json.loads((SAMPLES / "osm_overpass_fixture.json").read_text())
assert snapshot["savoroute_fixture"] is True
print("Input elements:", len(snapshot["elements"]))
print("First tags:", snapshot["elements"][0]["tags"])

## 2. Map source tags into the canonical catalog

Records without a name or location are skipped. A nearby element with a matching name is held for review rather than merged automatically. Item IDs are stable hashes of the OSM element identities and therefore cannot, by themselves, establish that two source records refer to the same establishment.

In [ ]:
result = import_osm_snapshot(snapshot, synthetic_fixture=True)
print("Admitted:", len(result.catalog.items))
print("Decisions:", dict(Counter(row.status for row in result.decisions)))
for row in result.decisions:
    print(row.source_ref, row.status, row.candidate_ids)

## 3. Inspect source mapping and search the resulting catalog

In schema v1, location is a display/search string. Coordinates and mapping evidence are kept in the import report. The description is a fixed, adapter-generated record label, not editorial text from a restaurant.

In [ ]:
report = result.report()
print("First accepted mapping:", report["admitted_sources"][0])
service = DiscoveryService(result.catalog)
for hit in service.search("Juniper", filters=SearchFilters(kind="restaurant"), limit=5):
    print(hit.item.title, hit.item.location, round(hit.score, 3))

## 4. Evaluate text extraction against reference annotations

The prior notebook generated provenance from the `labeled-text-v1` extractor. Here we compare that output to a separately stored gold file. The reference data is also synthetic, so a perfect result demonstrates only that the evaluation plumbing works.

In [ ]:
source_payload = json.loads((SAMPLES / "ingestion_sources.json").read_text())
gold = json.loads((SAMPLES / "extraction_gold.json").read_text())
batch = parse_source_batch(source_payload)
extracted = ingest_sources(batch, LabeledTextExtractor(), base_catalog=load_catalog())
provenance = extracted.provenance()
summary = evaluate_extraction(gold, provenance)
print("Records exact:", summary.exact_records, "/", summary.gold_records)
print("Micro F1:", summary.micro.f1)
for field, score in summary.fields.items():
    print(field, score.as_dict())

## 5. Perform a controlled error analysis

A changed cuisine prediction introduces a false positive and a false negative. This is exact-match scoring, not semantic equivalence: synonyms, alternative language, and ambiguous attributes require a separate annotation policy.

In [ ]:
modified = deepcopy(provenance)
modified["records"][0]["evidence"]["cuisines"][0]["value"] = "fusion"
changed = evaluate_extraction(gold, modified)
print("Micro F1 after the deliberate error:", round(changed.micro.f1, 3))
print("Cuisines:", changed.fields["cuisines"].as_dict())
print("Affected fields by source:", changed.errors)

## 6. Next validation stage

A real-source experiment should use a bounded, explicitly initiated Overpass download, preserve its raw snapshot and attribution, inspect every review candidate, and annotate a held-out set before interpreting extraction metrics. The notebook deliberately avoids network access. The CLI commands and licensing limits are documented in [`docs/osm-and-extraction-evaluation.md`](../docs/osm-and-extraction-evaluation.md).